First model submission (Model 1 – Random Forest non-linear regression)

Step 1 – Import Libraries

In [1]:
# Step 1: Imports for Model 1
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_log_error
from sklearn.preprocessing import LabelEncoder


Step 2 – Load Kaggle datasets

In [2]:
# Step 2: Load Kaggle data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

train.head()


,id,Sex,Length,Diameter,Height,Whole weight,Whole weight.1,Whole weight.2,Shell weight,Rings
0,0,F,0.550,0.430,0.150,0.7715,0.3285,0.1465,0.2400,11
1,1,F,0.630,0.490,0.145,1.1300,0.4580,0.2765,0.3200,11
2,2,I,0.160,0.110,0.025,0.0210,0.0055,0.0030,0.0050,6
3,3,M,0.595,0.475,0.150,0.9145,0.3755,0.2055,0.2500,10
4,4,I,0.555,0.425,0.130,0.7820,0.3695,0.1600,0.1975,9


Step 3 — Basic EDA + Missing Data Check

In [3]:
# Step 3: Quick EDA
print(train.info())
print(train.describe())

print("Missing values in train:")
print(train.isna().sum())

print("Missing values in test:")
print(test.isna().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90615 entries, 0 to 90614
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              90615 non-null  int64  
 1   Sex             90615 non-null  object 
 2   Length          90615 non-null  float64
 3   Diameter        90615 non-null  float64
 4   Height          90615 non-null  float64
 5   Whole weight    90615 non-null  float64
 6   Whole weight.1  90615 non-null  float64
 7   Whole weight.2  90615 non-null  float64
 8   Shell weight    90615 non-null  float64
 9   Rings           90615 non-null  int64  
dtypes: float64(7), int64(2), object(1)
memory usage: 6.9+ MB
None
                 id        Length      Diameter        Height  Whole weight  \
count  90615.000000  90615.000000  90615.000000  90615.000000  90615.000000   
mean   45307.000000      0.517098      0.401679      0.135464      0.789035   
std    26158.441658      0.118217      0.098026

Step 4 — Encode Categorical Variables

In [4]:
# Step 4: Encode categorical features BEFORE splitting
cat_cols = train.select_dtypes(include=["object"]).columns

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])
    label_encoders[col] = le


Step 5 — Create Features, Target, and Validation Split

In [5]:
# Step 5: Define features and target
target_col = "Rings"
X = train.drop(columns=[target_col])
y = train[target_col]

X_test = test.copy()

# Train/validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)


Step 6 — Fit Non‑Linear Model 1 (Random Forest)

In [6]:
# Step 6: Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Validation predictions + RMSLE
y_valid_pred = rf_model.predict(X_valid)
rmsle_rf = np.sqrt(mean_squared_log_error(y_valid, y_valid_pred))
print("Model 1 - Random Forest RMSLE (validation):", rmsle_rf)



Model 1 - Random Forest RMSLE (validation): 0.15486804498464388


Step 7 — Retrain Random Forest on Full Training Data

In [7]:
# Step 7: Retrain on full training data
rf_model_full = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf_model_full.fit(X, y)

# Predict on Kaggle test set
test_pred_rf = rf_model_full.predict(X_test)


Step 8 — Build First Submission File

In [8]:
# Step 8: Create first submission DataFrame
submission_rf = sample_submission.copy()
submission_rf["Rings"] = test_pred_rf

# Save CSV for Kaggle
submission_rf.to_csv("submission_model1_random_forest.csv", index=False)

submission_rf.head()


,id,Rings
0,90615,10.1825
1,90616,10.2125
2,90617,12.1300
3,90618,11.4075
4,90619,9.2925
